In [2]:
import pandas as pd 

In [3]:
import os

In [4]:
os.getcwd()

'c:\\ex\\week9\\1일차_실습'

In [7]:
df = pd.read_csv("../1일차_실습/01-03_열연_코일별_실습데이터.csv", parse_dates=["timestamp"])

In [8]:
REQUIRED_COLUMNS = {
    "timestamp",
    "coil_id",
    "coil_seq",
    "position_pct",
    "rolling_force_ton",
    "roll_gap_mm",
    "exit_thickness_mm",
    "target_thickness_mm",
    "main_motor_current_a",
    "furnace_temp_c",
    "roll_replaced",
}


In [9]:
df.columns

Index(['timestamp', 'coil_id', 'coil_seq', 'position_pct', 'rolling_force_ton',
       'roll_gap_mm', 'exit_thickness_mm', 'target_thickness_mm',
       'main_motor_current_a', 'furnace_temp_c', 'roll_replaced'],
      dtype='str')

In [10]:
missing_columns = REQUIRED_COLUMNS.difference(df.columns)

In [11]:
missing_columns

set()

In [12]:
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")
if df[["coil_id", "coil_seq"]].isna().any().any():
    raise ValueError("coil_id 또는 coil_seq에 결측값이 있습니다.")

In [13]:
df = df.sort_values(["coil_seq", "position_pct", "timestamp"])

In [14]:
df

,timestamp,coil_id,coil_seq,position_pct,rolling_force_ton,roll_gap_mm,exit_thickness_mm,target_thickness_mm,main_motor_current_a,furnace_temp_c,roll_replaced
0,2026-10-13 08:00:00,C001,1,0.0,914.1,2.0407,1.9857,2.0,1243.1,1242.2,0
1,2026-10-13 08:01:00,C001,1,3.4,912.1,2.0398,2.0065,2.0,1247.8,1244.1,0
2,2026-10-13 08:02:00,C001,1,6.9,916.6,2.0357,1.9991,2.0,1231.9,1245.2,0
3,2026-10-13 08:03:00,C001,1,10.3,905.3,2.0378,2.0064,2.0,1226.5,1234.9,0
4,2026-10-13 08:04:00,C001,1,13.8,917.2,2.0424,2.0004,2.0,1239.6,1241.7,0
...,...,...,...,...,...,...,...,...,...,...,...
1195,2026-10-14 03:55:00,C040,40,86.2,1023.2,2.0189,1.9884,2.0,1299.4,1235.8,0
1196,2026-10-14 03:56:00,C040,40,89.7,1023.4,2.0157,1.9796,2.0,1288.9,1222.5,0
1197,2026-10-14 03:57:00,C040,40,93.1,1027.0,2.0136,2.0030,2.0,1295.8,1236.8,0
1198,2026-10-14 03:58:00,C040,40,96.6,1033.3,2.0124,1.9957,2.0,1312.8,1249.7,0


In [15]:
print("=== 데이터 범위 ===")
print(f"행 수: {len(df):,} / 코일 수: {df['coil_id'].nunique()}")
print(
    "코일별 측정 행 수:",
    f"최소 {df.groupby('coil_id').size().min()}, "
    f"최대 {df.groupby('coil_id').size().max()}",
)


=== 데이터 범위 ===
행 수: 1,200 / 코일 수: 40
코일별 측정 행 수: 최소 30, 최대 30


In [16]:
print("\n=== 컬럼의 설비·역할·단위 ===")
signal_map = pd.DataFrame(
    [
        {
            "컬럼": "rolling_force_ton",
            "설비/위치": "열연 압연기(롤 스탠드)",
            "역할": "소재를 압하하는 데 필요한 압연 하중",
            "단위": "ton (데이터 표기)",
            "해석 주의": "소재·온도·압하 조건과 함께 변함",
        },
        {
            "컬럼": "roll_gap_mm",
            "설비/위치": "롤 갭 조정 기구(AGC 포함 가능)",
            "역할": "두께 제어를 위해 조정하는 조작량의 대리값",
            "단위": "mm",
            "해석 주의": "명령값인지 실제 갭 측정값인지는 이 CSV만으로 구분 불가",
        },
        {
            "컬럼": "exit_thickness_mm",
            "설비/위치": "압연기 출측 두께계",
            "역할": "출측 제품 두께(제어 대상이자 측정 피드백)",
            "단위": "mm",
            "해석 주의": "목표 두께 및 두께계 교정·상태와 비교",
        },
        {
            "컬럼": "target_thickness_mm",
            "설비/위치": "제품 규격/제어 설정",
            "역할": "목표 두께",
            "단위": "mm",
            "해석 주의": "출측 두께 오차를 계산할 기준",
        },
        {
            "컬럼": "main_motor_current_a",
            "설비/위치": "압연기 주전동기",
            "역할": "전동기 전류; 부하 상태를 간접적으로 보여주는 값",
            "단위": "A",
            "해석 주의": "전류 변화만으로 전동기 고장을 단정할 수 없음",
        },
        {
            "컬럼": "roll_replaced",
            "설비/위치": "롤 정비 이력",
            "역할": "롤 교체 이벤트 표시",
            "단위": "0/1",
            "해석 주의": "교체 시점만 제공하며 상세 정비 기록은 없음",
        },
    ]
)


=== 컬럼의 설비·역할·단위 ===


In [17]:
signal_map

,컬럼,설비/위치,역할,단위,해석 주의
0,rolling_force_ton,열연 압연기(롤 스탠드),소재를 압하하는 데 필요한 압연 하중,ton (데이터 표기),소재·온도·압하 조건과 함께 변함
1,roll_gap_mm,롤 갭 조정 기구(AGC 포함 가능),두께 제어를 위해 조정하는 조작량의 대리값,mm,명령값인지 실제 갭 측정값인지는 이 CSV만으로 구분 불가
2,exit_thickness_mm,압연기 출측 두께계,출측 제품 두께(제어 대상이자 측정 피드백),mm,목표 두께 및 두께계 교정·상태와 비교
3,target_thickness_mm,제품 규격/제어 설정,목표 두께,mm,출측 두께 오차를 계산할 기준
4,main_motor_current_a,압연기 주전동기,전동기 전류; 부하 상태를 간접적으로 보여주는 값,A,전류 변화만으로 전동기 고장을 단정할 수 없음
5,roll_replaced,롤 정비 이력,롤 교체 이벤트 표시,0/1,교체 시점만 제공하며 상세 정비 기록은 없음


In [18]:
df.columns

Index(['timestamp', 'coil_id', 'coil_seq', 'position_pct', 'rolling_force_ton',
       'roll_gap_mm', 'exit_thickness_mm', 'target_thickness_mm',
       'main_motor_current_a', 'furnace_temp_c', 'roll_replaced'],
      dtype='str')

In [19]:
summary = (
    df.groupby(["coil_seq", "coil_id"], as_index=False)
    .agg(
        sample_count=("exit_thickness_mm", "size"),
        rolling_force_mean_ton=("rolling_force_ton", "mean"),
        rolling_force_first_ton=("rolling_force_ton", "first"),
        rolling_force_last_ton=("rolling_force_ton", "last"),
        roll_gap_mean_mm=("roll_gap_mm", "mean"),
        exit_thickness_mean_mm=("exit_thickness_mm", "mean"),
        exit_thickness_std_mm=("exit_thickness_mm", "std"),
        target_thickness_mean_mm=("target_thickness_mm", "mean"),
        motor_current_mean_a=("main_motor_current_a", "mean"),
        furnace_temp_mean_c=("furnace_temp_c", "mean"),
        roll_replaced=("roll_replaced", "max"),
    )
    .sort_values("coil_seq")
    .reset_index(drop=True)
)

In [20]:
summary

,coil_seq,coil_id,sample_count,rolling_force_mean_ton,rolling_force_first_ton,rolling_force_last_ton,roll_gap_mean_mm,exit_thickness_mean_mm,exit_thickness_std_mm,target_thickness_mean_mm,motor_current_mean_a,furnace_temp_mean_c,roll_replaced
0,1,C001,30,921.736667,914.1,929.9,2.040237,1.998957,0.007828,2.0,1247.476667,1234.686667,0
1,2,C002,30,925.036667,908.1,936.7,2.039163,1.999660,0.007261,2.0,1249.220000,1233.466667,0
2,3,C003,30,930.553333,930.7,937.6,2.035623,2.000173,0.006468,2.0,1252.896667,1234.663333,0
3,4,C004,30,935.683333,929.7,952.1,2.036460,1.999290,0.007635,2.0,1256.676667,1234.793333,0
4,5,C005,30,940.880000,926.6,952.1,2.035033,2.000623,0.008932,2.0,1256.453333,1236.200000,0
5,6,C006,30,944.356667,927.6,955.6,2.033170,1.998953,0.008097,2.0,1256.653333,1237.476667,0
6,7,C007,30,952.596667,943.7,964.2,2.030493,1.998627,0.006786,2.0,1266.286667,1235.306667,0
7,8,C008,30,954.783333,940.4,955.8,2.029093,2.000310,0.009434,2.0,1265.646667,1236.856667,0
8,9,C009,30,963.033333,955.4,969.2,2.028113,2.000473,0.009018,2.0,1271.986667,1235.753333,0
9,10,C010,30,966.466667,954.8,978.5,2.026473,2.000300,0.007868,2.0,1273.020000,1237.723333,0


In [ ]:
summary["thickness_mean_error_mm"] = (
    summary["exit_thickness_mean_mm"] - summary["target_thickness_mean_mm"]
)

In [22]:
summary["thickness_mean_error_mm"]

0    -0.001043
1    -0.000340
2     0.000173
3    -0.000710
4     0.000623
5    -0.001047
6    -0.001373
7     0.000310
8     0.000473
9     0.000300
10   -0.003340
11    0.002440
12   -0.000307
13    0.001113
14   -0.001227
15    0.000170
16    0.001753
17   -0.000663
18    0.000053
19    0.000120
20    0.001213
21    0.000183
22   -0.000167
23   -0.000047
24    0.000950
25   -0.002543
26   -0.000240
27    0.000007
28    0.001583
29    0.001153
30    0.003613
31    0.009437
32   -0.003490
33    0.005023
34    0.008833
35   -0.003257
36   -0.006880
37    0.006990
38    0.003403
39   -0.000800
Name: thickness_mean_error_mm, dtype: float64

In [23]:
summary["force_change_from_previous_coil_ton"] = (
    summary["rolling_force_mean_ton"].diff()
)

In [24]:
print("\n=== 코일별 평균·표준편차 ===")
print(
    summary[
        [
            "coil_seq",
            "coil_id",
            "sample_count",
            "rolling_force_mean_ton",
            "roll_gap_mean_mm",
            "exit_thickness_mean_mm",
            "exit_thickness_std_mm",
            "thickness_mean_error_mm",
            "motor_current_mean_a",
            "roll_replaced",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


=== 코일별 평균·표준편차 ===
 coil_seq coil_id  sample_count  rolling_force_mean_ton  roll_gap_mean_mm  exit_thickness_mean_mm  exit_thickness_std_mm  thickness_mean_error_mm  motor_current_mean_a  roll_replaced
        1    C001            30                921.7367            2.0402                  1.9990                 0.0078                  -0.0010             1247.4767              0
        2    C002            30                925.0367            2.0392                  1.9997                 0.0073                  -0.0003             1249.2200              0
        3    C003            30                930.5533            2.0356                  2.0002                 0.0065                   0.0002             1252.8967              0
        4    C004            30                935.6833            2.0365                  1.9993                 0.0076                  -0.0007             1256.6767              0
        5    C005            30                940.8800         

In [25]:
print("\n=== 하중이 직전 코일보다 낮아진 지점 ===")
force_reversals = summary.loc[
    summary["force_change_from_previous_coil_ton"] < 0,
    [
        "coil_seq",
        "coil_id",
        "force_change_from_previous_coil_ton",
        "rolling_force_mean_ton",
        "roll_replaced",
    ],
]


=== 하중이 직전 코일보다 낮아진 지점 ===


In [26]:
force_reversals

,coil_seq,coil_id,force_change_from_previous_coil_ton,rolling_force_mean_ton,roll_replaced
20,21,C021,-98.626667,920.033333,1


In [27]:
if force_reversals.empty:
    print("직전 코일보다 평균 하중이 낮아진 지점이 없습니다.")
else:
    print(force_reversals.round(3).to_string(index=False))
    confirmed_replacements = force_reversals.loc[
        force_reversals["roll_replaced"] == 1, "coil_seq"
    ].tolist()
    if confirmed_replacements:
        print(
            "롤 교체 표시와 하중 하락이 함께 나타난 코일:",
            ", ".join(map(str, confirmed_replacements)),
        )

 coil_seq coil_id  force_change_from_previous_coil_ton  rolling_force_mean_ton  roll_replaced
       21    C021                              -98.627                 920.033              1
롤 교체 표시와 하중 하락이 함께 나타난 코일: 21


In [28]:
reference = summary.loc[summary["coil_seq"].between(1, 30), "exit_thickness_std_mm"]

In [29]:
reference

0     0.007828
1     0.007261
2     0.006468
3     0.007635
4     0.008932
5     0.008097
6     0.006786
7     0.009434
8     0.009018
9     0.007868
10    0.006404
11    0.006363
12    0.008358
13    0.009050
14    0.008296
15    0.009131
16    0.007263
17    0.008022
18    0.006921
19    0.007799
20    0.008632
21    0.007208
22    0.006382
23    0.007280
24    0.007501
25    0.008995
26    0.007464
27    0.008378
28    0.008215
29    0.008232
Name: exit_thickness_std_mm, dtype: float64

In [30]:
if reference.empty or reference.isna().any():
    raise ValueError("두께 변동 기준을 계산할 비교 구간이 부족합니다.")

In [31]:
std_limit_mm = reference.mean() + 3 * reference.std()
summary["thickness_variation_flag"] = (
    summary["exit_thickness_std_mm"] > std_limit_mm
)

In [32]:
flagged = summary.loc[summary["thickness_variation_flag"], "coil_seq"]
first_flagged_coil = int(flagged.min()) if not flagged.empty else None

In [33]:
print("\n=== 출측 두께 변동 구간 ===")
print(
    "코일 1~30 비교 구간의 코일별 두께 표준편차 평균:",
    f"{reference.mean():.4f} mm",
)


=== 출측 두께 변동 구간 ===
코일 1~30 비교 구간의 코일별 두께 표준편차 평균: 0.0078 mm


In [34]:
print(
    "탐색 기준(비교 구간 평균 + 3 표준편차):",
    f"{std_limit_mm:.4f} mm (현장 경보 기준 아님)",
)

탐색 기준(비교 구간 평균 + 3 표준편차): 0.0105 mm (현장 경보 기준 아님)


In [35]:
if first_flagged_coil is None:
    print("탐색 기준을 넘는 코일이 없습니다.")
else:
    print(
        f"처음 기준을 넘는 코일: {first_flagged_coil} "
        f"(C{first_flagged_coil:03d}); 이후 기준 초과 코일: "
        f"{', '.join(map(str, flagged.tolist()))}"
    )
    before = summary.loc[summary["coil_seq"].between(1, first_flagged_coil - 1)]
    after = summary.loc[summary["coil_seq"].ge(first_flagged_coil)]
    print(
        "전후 평균 코일별 두께 표준편차:",
        f"{before['exit_thickness_std_mm'].mean():.4f} mm → "
        f"{after['exit_thickness_std_mm'].mean():.4f} mm",
    )
    print(
        "전후 평균 코일 출측 두께의 목표 오차 절대값:",
        f"{before['thickness_mean_error_mm'].abs().mean():.4f} mm → "
        f"{after['thickness_mean_error_mm'].abs().mean():.4f} mm",
    )


처음 기준을 넘는 코일: 31 (C031); 이후 기준 초과 코일: 31, 32, 33, 34, 35, 36, 37, 38, 39, 40
전후 평균 코일별 두께 표준편차: 0.0078 mm → 0.0239 mm
전후 평균 코일 출측 두께의 목표 오차 절대값: 0.0009 mm → 0.0052 mm


In [36]:
print("\n=== 운전·제어 해석 ===")
print(
    "제어 관점: 목표 두께와 출측 두께 측정값을 비교하고, "
    "AGC가 롤 갭 조정으로 오차를 줄이는 구조로 이해합니다."
)
print(
    "다만 roll_gap_mm가 AGC 명령값인지 실제 갭 측정값인지는 "
    "태그 정의서나 제어 시스템 자료로 확인해야 합니다."
)
print(
    "main_motor_current_a는 부하 상태의 간접 지표입니다. "
    "압연 하중과 함께 보되 전류만으로 모터 고장을 진단하지 않습니다."
)


=== 운전·제어 해석 ===
제어 관점: 목표 두께와 출측 두께 측정값을 비교하고, AGC가 롤 갭 조정으로 오차를 줄이는 구조로 이해합니다.
다만 roll_gap_mm가 AGC 명령값인지 실제 갭 측정값인지는 태그 정의서나 제어 시스템 자료로 확인해야 합니다.
main_motor_current_a는 부하 상태의 간접 지표입니다. 압연 하중과 함께 보되 전류만으로 모터 고장을 진단하지 않습니다.


In [37]:
print("\n=== 정비·추가 증거 확인 ===")
replacement_coils = summary.loc[
    summary["roll_replaced"] == 1, ["coil_seq", "coil_id", "rolling_force_mean_ton"]
]
if replacement_coils.empty:
    print("CSV에서 확인된 롤 교체 표시가 없습니다.")
else:
    print("CSV의 롤 교체 표시:")
    print(replacement_coils.round(3).to_string(index=False))
print(
    "이 CSV에는 roll_replaced 외의 정비 상세, AGC 명령·보정량·알람, "
    "두께계 교정 이력이 없습니다."
)
print(
    "원인 판정 전 확인할 자료: 길이 위치별 AGC 갭 명령/실제값·보정량·알람, "
    "롤 교체·연삭 기록, 소재 강종·폭·두께·온도, 두께계 교정·품질 이력."
)


=== 정비·추가 증거 확인 ===
CSV의 롤 교체 표시:
 coil_seq coil_id  rolling_force_mean_ton
       21    C021                 920.033
이 CSV에는 roll_replaced 외의 정비 상세, AGC 명령·보정량·알람, 두께계 교정 이력이 없습니다.
원인 판정 전 확인할 자료: 길이 위치별 AGC 갭 명령/실제값·보정량·알람, 롤 교체·연삭 기록, 소재 강종·폭·두께·온도, 두께계 교정·품질 이력.
